# Практическая работа 2. Работа с LLM через API
**Студент:** Павлов Кирилл Кириллович  
**Группа:** 15.27Д-ИСТ05/26Б  
**Вариант:** 18

In [1]:
import json
from pathlib import Path
import pandas as pd
from rental_llm import run_experiment, evaluate_counts
RUN_API = False

In [2]:
if RUN_API:
    status = run_experiment(advanced=True, with_metrics=True)
else:
    status = json.loads(Path('results/metrics.json').read_text(encoding='utf-8'))
print(json.dumps(status, ensure_ascii=False, indent=2))

{
  "variant": 18,
  "total_requests": 15,
  "model": "GigaChat-2",
  "status": "partial",
  "advanced": true,
  "successful_basic": 15,
  "basic": {
    "total": 15,
    "correct": 15,
    "errors": 0,
    "accuracy": 1.0
  },
  "structured": {
    "fields": {
      "count": {
        "correct": 11,
        "total": 15,
        "accuracy": 0.7333333333333333
      },
      "count_adults": {
        "correct": 11,
        "total": 15,
        "accuracy": 0.7333333333333333
      },
      "count_children": {
        "correct": 11,
        "total": 15,
        "accuracy": 0.7333333333333333
      },
      "start_date": {
        "correct": 8,
        "total": 15,
        "accuracy": 0.5333333333333333
      },
      "nights": {
        "correct": 8,
        "total": 15,
        "accuracy": 0.5333333333333333
      },
      "price_per_day": {
        "correct": 10,
        "total": 15,
        "accuracy": 0.6666666666666666
      }
    },
    "mean_extra_accuracy": 0.6399999999999999,
   

In [3]:
df = pd.read_csv('results/rental_with_results.csv')
display(df[['amount','text','result','error']])
metric = evaluate_counts(df)
print(f"Верно: {metric['correct']} из {metric['total']}")
print(f"Ошибок: {metric['errors']}; точность: {metric['accuracy']:.1%}")

,amount,text,result,error
0,4,Снимем жильё с 30 июня по 7 июля 3 взрослых и ...,4,NaN
1,5,"Ищем жилье! 2взрослых и 3 ребенка ( 6,6 и 4 го...",5,NaN
2,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,NaN
3,2,"Ищем жилье в самом Лазаревском для двоих, 2-ме...",2,NaN
4,3,"Ищем жильё для 2-х взрослых, ребёнок (7 лет) и...",3,NaN
5,3,"Ищу жилье, интересует Пицунда, песчаный пляж, ...",3,NaN
6,4,"Лазаревское, Солоники - бюджетное жилье на 3 с...",4,NaN
7,3,Ищем жильё с 18-19 августа на 7 дней.Частный с...,3,NaN
8,4,Здравствуйте! Ищем жильё на августа числа с 6-...,4,NaN
9,3,Добрый день!Ищем жилье с 25.04-05.05 2 взрослы...,3,NaN


Верно: 15 из 15
Ошибок: 0; точность: 100.0%


## Задание 2. Few-shot и JSON
Поля: `count`, `count_adults`, `count_children`, `start_date`, `nights`, `price_per_day`, `remarks`.
Эталон: `rental_18_labels.json`. Правила разметки: `ANNOTATION_RULES.md`.

In [4]:
from rental_llm import evaluate_structured
records = json.loads(Path('results/structured_results.json').read_text(encoding='utf-8'))
labels = json.loads(Path('rental_18_labels.json').read_text(encoding='utf-8'))
quality = evaluate_structured(records, labels)
display(pd.DataFrame(quality['fields']).T)
print(f"Средняя по пяти дополнительным полям: {quality['mean_extra_accuracy']:.1%}")
print(f"Средняя по всем шести показателям: {quality['mean_all_accuracy']:.1%}")
print('Базовый count:', f"{metric['accuracy']:.1%}")
print('Few-shot count:', f"{quality['fields']['count']['accuracy']:.1%}")

,correct,total,accuracy
count,11.0,15.0,0.733333
count_adults,11.0,15.0,0.733333
count_children,11.0,15.0,0.733333
start_date,8.0,15.0,0.533333
nights,8.0,15.0,0.533333
price_per_day,10.0,15.0,0.666667


Средняя по пяти дополнительным полям: 64.0%
Средняя по всем шести показателям: 65.6%
Базовый count: 100.0%
Few-shot count: 73.3%


In [5]:
display(pd.DataFrame([{'id':r['id'],'status':r['status'],**(r['parsed'] or {'error':r['error']})} for r in records]))

,id,status,count,count_adults,count_children,start_date,nights,price_per_day,remarks,error
0,1,ok,4.0,3.0,1.0,--06-30,8.0,NaN,,NaN
1,2,ok,5.0,2.0,3.0,2023-07-11,12.0,NaN,"2 двухспалки,недалеко от центра и моря,эконом",NaN
2,3,ok,6.0,2.0,4.0,2018-05-29,11.0,NaN,"с кухней, недалёко от моря",NaN
3,4,ok,2.0,2.0,0.0,--09-07,7.0,700.0,,NaN
4,5,ok,3.0,2.0,1.0,--07-15,10.0,NaN,"Только Чёрное море, не Крым и Анапа. Собака.",NaN
5,6,error,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OutputParserException: запрос или валидация от...
6,7,ok,4.0,4.0,0.0,2023-09-05,3.0,NaN,,NaN
7,8,ok,3.0,3.0,0.0,--08-18,7.0,NaN,Частный сектор.Обязательно с местом для парков...,NaN
8,9,ok,4.0,2.0,2.0,--08-06,7.0,10000.0,,NaN
9,10,error,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OutputParserException: запрос или валидация от...
